# Vector Index EDA

Read-only exploration for Feature 07 vector indexing. Run `uv run -m uni_rag_agent index vector` after extraction and data-summary chunks exist, then use this notebook to inspect ChromaDB / vector coverage recorded in `embeddings`, model/dimension consistency, collection sizes, and missing or stale embeddings.

Safety boundary: this notebook reads generated app data only. It opens SQLite in read-only mode, enables `PRAGMA query_only = ON`, opens ChromaDB read-only, and must not mutate SQLite, ChromaDB files, indexes, `Courses/`, or source course files. Clear outputs and execution counts before committing.

In [ ]:
from pathlib import Path
import sqlite3

import matplotlib.pyplot as plt
import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "context"
        ).is_dir():
            return candidate
    raise RuntimeError(
        "Could not find repo root. Start Jupyter from this project or notebooks/."
    )


repo_root = find_repo_root()
sqlite_path = repo_root / "data" / "uni_rag.sqlite"
chroma_dir = repo_root / "data" / "indexes" / "vector"
if not sqlite_path.is_file():
    raise FileNotFoundError(
        f"SQLite database not found: {sqlite_path}. Run storage, inventory, "
        "extraction, and vector indexing first."
    )

connection = sqlite3.connect(f"{sqlite_path.resolve().as_uri()}?mode=ro", uri=True)
connection.execute("PRAGMA query_only = ON")

In [ ]:
embeddings = pd.read_sql_query(
    """
    SELECT
        embeddings.chunk_id,
        embeddings.embedding_model,
        embeddings.embedding_dim,
        embeddings.vector_backend,
        embeddings.vector_collection,
        embeddings.vector_id,
        chunks.source_type,
        files.path AS file_path,
        files.index_status,
        courses.name AS course_name
    FROM embeddings
    JOIN chunks ON chunks.id = embeddings.chunk_id
    JOIN files ON files.id = chunks.file_id
    LEFT JOIN courses ON courses.id = files.course_id
    """,
    connection,
)

eligible_chunks = pd.read_sql_query(
    """
    SELECT chunks.id AS chunk_id, chunks.source_type
    FROM chunks
    JOIN files ON files.id = chunks.file_id
    WHERE files.index_status = 'indexed'
      AND TRIM(chunks.text) <> ''
      AND chunks.source_type IN ('document', 'slides', 'notebook', 'code', 'data_schema', 'transcript')
    """,
    connection,
)

distinct_models = (
    int(embeddings["embedding_model"].nunique()) if not embeddings.empty else 0
)
pd.DataFrame(
    [
        {"metric": "eligible_chunks", "count": len(eligible_chunks)},
        {"metric": "embedding_rows", "count": len(embeddings)},
        {"metric": "distinct_models", "count": distinct_models},
    ]
)

In [ ]:
eligible_by_source = (
    eligible_chunks.groupby("source_type").size().rename("eligible_chunks")
)
embedded_by_source = embeddings.groupby("source_type").size().rename("embedding_rows")
source_coverage = (
    pd.concat([eligible_by_source, embedded_by_source], axis=1)
    .fillna(0)
    .astype(int)
    .reset_index()
    .sort_values("source_type")
)

# Eligible chunks that are missing an embedding for each indexed model.
eligible_ids = set(eligible_chunks["chunk_id"])
models = sorted(embeddings["embedding_model"].unique()) if not embeddings.empty else []
missing_rows = []
for model in models:
    embedded_ids = set(
        embeddings.loc[embeddings["embedding_model"] == model, "chunk_id"]
    )
    missing_rows.append(
        {
            "embedding_model": model,
            "missing_embeddings": len(eligible_ids - embedded_ids),
        }
    )
missing_embeddings = pd.DataFrame(
    missing_rows or [{"embedding_model": "(none)", "missing_embeddings": 0}]
)

# Stale mapping rows: an embedding whose source file is no longer indexed.
stale_embeddings = embeddings[embeddings["index_status"] != "indexed"]
coverage_summary = pd.DataFrame(
    [
        {"metric": "eligible_chunks", "count": len(eligible_chunks)},
        {"metric": "embedding_rows", "count": len(embeddings)},
        {"metric": "stale_embedding_rows", "count": len(stale_embeddings)},
    ]
)
coverage_summary

In [ ]:
# ChromaDB inspection is optional and read-only; it degrades gracefully if the
# vector store or the chromadb package is unavailable.
collection_summary = pd.DataFrame(columns=["collection", "chroma_count"])
try:
    import chromadb
    from chromadb.config import Settings

    if chroma_dir.is_dir():
        client = chromadb.PersistentClient(
            path=str(chroma_dir),
            settings=Settings(anonymized_telemetry=False),
        )
        rows = []
        for collection in client.list_collections():
            name = collection if isinstance(collection, str) else collection.name
            rows.append(
                {
                    "collection": name,
                    "chroma_count": client.get_collection(name=name).count(),
                }
            )
        if rows:
            collection_summary = pd.DataFrame(rows).sort_values("collection")
    else:
        print(f"ChromaDB directory not found: {chroma_dir}")
except Exception as exc:  # pragma: no cover - notebook diagnostic only
    print(f"Skipping ChromaDB inspection: {exc}")

sqlite_collection_counts = (
    embeddings.groupby("vector_collection").size().rename("sqlite_rows").reset_index()
    if not embeddings.empty
    else pd.DataFrame(columns=["vector_collection", "sqlite_rows"])
)
sqlite_collection_counts

In [ ]:
if not embeddings.empty:
    model_consistency = (
        embeddings.groupby("embedding_model")
        .agg(
            embedding_rows=("chunk_id", "size"),
            distinct_dims=("embedding_dim", "nunique"),
            distinct_collections=("vector_collection", "nunique"),
            non_indexed_sources=("index_status", lambda s: int((s != "indexed").sum())),
        )
        .reset_index()
    )
else:
    model_consistency = pd.DataFrame(
        columns=[
            "embedding_model",
            "embedding_rows",
            "distinct_dims",
            "distinct_collections",
            "non_indexed_sources",
        ]
    )
model_consistency

In [ ]:
if not source_coverage.empty:
    ax = source_coverage.set_index("source_type")[
        ["eligible_chunks", "embedding_rows"]
    ].plot(kind="bar")
    ax.set_title("Vector index coverage by source type")
    ax.set_xlabel("source_type")
    ax.set_ylabel("rows")
    plt.tight_layout()
else:
    print("No vector source coverage rows to plot.")

In [ ]:
if not missing_embeddings.empty:
    ax = missing_embeddings.set_index("embedding_model")["missing_embeddings"].plot(
        kind="bar"
    )
    ax.set_title("Missing embeddings per model")
    ax.set_xlabel("embedding_model")
    ax.set_ylabel("eligible chunks without an embedding")
    plt.tight_layout()
else:
    print("No missing-embedding rows to plot.")

In [ ]:
if not embeddings.empty:
    ax = embeddings.groupby("embedding_model").size().plot(kind="bar")
    ax.set_title("Embedding rows by model")
    ax.set_xlabel("embedding_model")
    ax.set_ylabel("rows")
    plt.tight_layout()
else:
    print("No embedding rows to plot.")